# CSA17 Artificial Intelligence - Assignment 1 (CO1)
Athisaya U - 192571001

Hospital medicine-delivery robot: BFS, DFS and UCS. The Task 8 benchmark source is run first because Tasks 1-7 use its classes and results.

## Task 8 - benchmark source (setup for all tasks)

In [1]:
import random, time, statistics, heapq, json, platform
from collections import deque
from array import array

SIZES = (1000, 10000, 100000, 1000000)
SEED = 17
RUNS = 5
PREDICTION = ("The direct lift will give BFS one hop but excess cost. "
              "DFS should store few active frames. UCS should recover the minimum cost. "
              "A side sensor should reduce worst-case docking moves.")
print("Before running:", PREDICTION, flush=True)

class Hospital:
    def __init__(self, n):
        self.n = n
        self.ward = n - 2
        self.weights = random.Random(SEED).randbytes(n)

    def edges(self, u):
        # Last vertex is an isolated, closed ward. Costs are positive time units.
        for v in (2*u + 1, 2*u + 2):
            if v <= self.ward:
                yield v, 1 + self.weights[v] % 8
        if u == 0:
            yield self.ward, 1000

def finish(parent, s, g):
    if parent[g] == -1:
        return []
    path = [g]
    while path[-1] != s:
        path.append(parent[path[-1]])
    return path[::-1]

def search(h, s, g, name):
    parent = array('i', [-1]) * h.n
    parent[s] = s
    generated, expanded, peak = 1, 0, 1
    if name == 'DFS':
        def visit(u, level):
            nonlocal generated, expanded, peak
            peak = max(peak, level)
            if u == g:
                return True
            expanded += 1
            for v, w in h.edges(u):
                if parent[v] == -1:
                    generated += 1
                    parent[v] = u
                    if visit(v, level + 1):
                        return True
            return False
        visit(s, 1)
    elif name == 'BFS':
        q = deque([s])
        while q:
            u = q.popleft()
            if u == g:
                break
            expanded += 1
            for v, w in h.edges(u):
                if parent[v] == -1:
                    parent[v] = u
                    q.append(v)
                    generated += 1
            peak = max(peak, len(q))
    else:
        assert name == 'UCS'
        dist = array('q', [-1]) * h.n
        dist[s] = 0
        q = [(0,s)]
        while q:
            cost,u = heapq.heappop(q)
            if cost != dist[u]:
                continue
            if u == g:
                break
            expanded += 1
            for v,w in h.edges(u):
                new = cost + w
                if dist[v] == -1 or new < dist[v]:
                    dist[v],parent[v] = new,u
                    heapq.heappush(q,(new,v))
                    generated += 1
            peak = max(peak,len(q))
    return finish(parent,s,g), (generated,expanded,peak)

def cost_of(h,path):
    return sum(dict(h.edges(u))[v] for u,v in zip(path,path[1:]))

def minimum(h):
    # Independent oracle: compare the unique tree path with the lift.
    u, cost = h.ward, 0
    while u:
        cost += 1 + h.weights[u] % 8
        u = (u-1)//2
    return min(cost,1000)

def experiment():
    rows, prep = [], {}
    for n in SIZES:
        builds = []
        for _ in range(RUNS):
            t = time.perf_counter()
            h = Hospital(n)
            builds.append(time.perf_counter()-t)
        prep[n] = statistics.mean(builds)*1000
        for name in ('BFS','DFS','UCS'):
            times = []
            for _ in range(RUNS):
                t = time.perf_counter()
                path, stats = search(h,0,h.ward,name)
                times.append((time.perf_counter()-t)*1000)
                assert path[0] == 0 and path[-1] == h.ward
                cost = cost_of(h,path)
                if name == 'UCS': assert cost == minimum(h)
                if name == 'BFS': assert len(path) == 2
            for goal,expected in ((n-1,[]),(0,[0])):
                assert search(h,0,goal,name)[0] == expected
            row = dict(n=n,method=name,gen=stats[0],exp=stats[1],peak=stats[2],
                       depth=len(path)-1,cost=cost,opt=minimum(h),path=path,
                       ms=statistics.mean(times),runs=times)
            rows.append(row)
            print(n,name,stats,len(path)-1,cost,round(row['ms'],3),
                  'unreachable=[]; start=goal=[0]',flush=True)
    return rows,prep

rows, prep = experiment()

Before running: The direct lift will give BFS one hop but excess cost. DFS should store few active frames. UCS should recover the minimum cost. A side sensor should reduce worst-case docking moves.


1000 BFS (8, 3, 5) 1 1000 0.008 unreachable=[]; start=goal=[0]


1000 DFS (977, 976, 10) 9 32 0.378 unreachable=[]; start=goal=[0]


1000 UCS (596, 361, 245) 9 32 0.292 unreachable=[]; start=goal=[0]


10000 BFS (8, 3, 5) 1 1000 0.005 unreachable=[]; start=goal=[0]


10000 DFS (3621, 3620, 14) 13 50 1.388 unreachable=[]; start=goal=[0]


... (3 outputs left out)

100000 UCS (99576, 97929, 19440) 16 85 111.17 unreachable=[]; start=goal=[0]


1000000 BFS (8, 3, 5) 1 1000 0.544 unreachable=[]; start=goal=[0]


1000000 DFS (951431, 951430, 20) 19 94 400.499 unreachable=[]; start=goal=[0]


1000000 UCS (972654, 921808, 185968) 19 94 1980.552 unreachable=[]; start=goal=[0]


## Task 1

In [2]:
h = Hospital(1000)
for method in ('BFS', 'DFS', 'UCS'):
    path, _ = search(h, 0, h.ward, method)
    print(method, len(path)-1, 'moves;', cost_of(h,path), 'time units')

BFS 1 moves; 1000 time units
DFS 9 moves; 32 time units
UCS 9 moves; 32 time units


## Task 2

In [3]:
for method in ('BFS', 'DFS', 'UCS'):
    _, counts = search(Hospital(1000),0,998,method)
    print(method, 'admitted / expanded / peak:', *counts)

BFS admitted / expanded / peak: 8 3 5
DFS admitted / expanded / peak: 977 976 10
UCS admitted / expanded / peak: 596 361 245


## Task 3

In [4]:
for depth in (3, 6, 9):
    levels = [2**i for i in range(depth+1)]
    print(depth, 'levels total', sum(levels), 'leaf width', levels[-1])

3 levels total 15 leaf width 8
6 levels total 127 leaf width 64
9 levels total 1023 leaf width 512


## Task 4

In [5]:
h = Hospital(1000)
for goal in (0, h.ward, h.n-1):
    print(goal, [search(h,0,goal,m)[1][1] for m in ('BFS','DFS','UCS')])

0 [0, 0, 0]
998 [3, 976, 361]
999 [999, 999, 999]


## Task 5

In [6]:
for r in rows:
    if r['method'] == 'DFS':
        print(r['n'], 'states;', r['peak'], 'active calls')

1000 states; 10 active calls
10000 states; 14 active calls
100000 states; 17 active calls
1000000 states; 20 active calls


## Task 6

In [7]:
def calls(h,b):
    if h == 0:
        return 1
    return 1 + sum(calls(h-1,b) for _ in range(b))
for height in (2,3,4):
    print(height, calls(height,2))

2 7
3 15
4 31


## Task 7

In [8]:
for depth in (2,3,4):
    direct = sum((depth-i+1)*2**i for i in range(depth+1))
    repeated = sum(calls(h,2) for h in range(depth+1))
    assert direct == repeated
    print(depth,'single-pass',calls(depth,2),'iterative deepening',direct)

2 single-pass 7 iterative deepening 11
3 single-pass 15 iterative deepening 26
4 single-pass 31 iterative deepening 57


## Task 8 - cyclic-map check

In [9]:
class LoopMap:
    n = 5
    def edges(self,u):
        # 0 pharmacy, 1 lift lobby, 2 ramp, 3 corridor, 4 ward
        return iter((((1,2),(2,1)),((0,2),(4,20)),((3,1),),
                     ((4,1),),((1,20),))[u])

for name in ('BFS','DFS','UCS'):
    h = LoopMap()
    path,stats = search(h,0,4,name)
    cost = cost_of(h,path)
    assert cost == (3 if name == 'UCS' else 22)
    print(name,path,'cost',cost,'generated/expanded/peak',stats)

BFS [0, 1, 4] cost 22 generated/expanded/peak (5, 3, 2)
DFS [0, 1, 4] cost 22 generated/expanded/peak (3, 2, 3)
UCS [0, 2, 3, 4] cost 3 generated/expanded/peak (6, 4, 2)


## Task 9

In [10]:
for r in rows:
    if r['n'] == 1000000:
        print(r['method'], r['gen'], r['exp'], r['peak'],
              r['depth'], r['cost'], round(r['ms'],3))

BFS 8 3 5 1 1000 0.544
DFS 951431 951430 20 19 94 400.499
UCS 972654 921808 185968 19 94 1980.552


## Task 10

In [11]:
for r in rows:
    if r['method'] == 'UCS':
        print(r['n'], round(100*r['exp']/r['n'],1), '% expanded')

1000 36.1 % expanded
10000 50.0 % expanded
100000 97.9 % expanded
1000000 92.2 % expanded


## Task 11

In [12]:
million = [r for r in rows if r['n'] == 1000000]
lift, dfs, ucs = million
print('BFS cost', lift['cost'], 'UCS cost', ucs['cost'])
print('Travel-time units saved', lift['cost'] - ucs['cost'])

BFS cost 1000 UCS cost 94
Travel-time units saved 906


## Task 12

In [13]:
def dock(k,sensor):
    m = k//2
    initial = frozenset(range(-m,m+1)) - {0}
    def step(B,a):
        return frozenset(max(-m,min(m,x+a)) for x in B)
    if not sensor:
        q = deque([(initial,'')])
        seen = {initial}
        while q:
            B,path = q.popleft()
            if B == {0}:
                for x in initial:
                    loc = {x}
                    for a in path:
                        loc = step(loc,1 if a=='R' else -1)
                    assert loc == {0}
                return len(seen),len(path),path
            for a,letter in ((-1,'L'),(1,'R')):
                C = step(B,a)
                if C not in seen:
                    seen.add(C)
                    q.append((C,path+letter))
    # Count distinct information sets reached while executing a fixed policy.
    seen = {initial}
    def follow(B):
        branches = [frozenset(x for x in B if (x>0)-(x<0)==sign)
                    for sign in (-1,0,1)]
        longest = 0
        for group in filter(None,branches):
            seen.add(group)
            if group == {0}: continue
            C = step(group,1 if min(group)<0 else -1)
            seen.add(C)
            longest = max(longest,1+follow(C))
        return longest
    length = follow(initial)
    assert length == m
    return len(seen),length,'L if right; R if left; stop at dock'

for k in (2,4,8):
    for sensed in (False,True):
        print(k, sensed, *dock(k,sensed))

2 False 6 3 LLR
2 True 4 1 L if right; R if left; stop at dock
4 False 20 6 LLLLRR
4 True 8 2 L if right; R if left; stop at dock
8 False 99 12 LLLLLLLLRRRR
8 True 16 4 L if right; R if left; stop at dock
